# Set Up

## A. Library

Menginstall Library: (1) Pytorch dan (2) Transformer

In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
%pip install transformers

Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Mengimport Library

In [ ]:
import os
import torch
import logging
import warnings
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.patches as mpatches

## B. Device

Melakukan Set Up Device:<br>(1) Mendeteksi Hardware,<br>(2) Melakukan Konfigurasi Device yang Akan Digunakan Dalam Eksperimen ini,<br>(3) Memastikan Komputasi Berjalan di GPU

In [ ]:
# =========================================================
# 1. SETUP LIBRARY & DEVICE
# =========================================================

print("-" * 50)
# Logika pendeteksian perangkat
# Jika CUDA (GPU NVIDIA) tersedia, gunakan 'cuda'. Jika tidak, pakai 'cpu'.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device yang digunakan: {device}")

if torch.cuda.is_available():
    print(f"Nama GPU: {torch.cuda.get_device_name(0)}")
    print(f"Versi CUDA: {torch.version.cuda}")
else:
    print("Peringatan: GPU tidak terdeteksi. Proses akan berjalan lambat di CPU.")
print("-" * 50)

os.environ["TRANSFORMERS_OFFLINE"] = "0"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["DISABLE_SAFETENSORS_CONVERSION"] = "1"

--------------------------------------------------
Device yang digunakan: cuda
Nama GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Versi CUDA: 12.4
--------------------------------------------------


## C. Model

### 1. Load Model IndoBERT (Versi IndoLEM)

Load Model IndoBERT Versi IndoLEM (Koto et al., 2020)

In [ ]:
print("Melakukan Loading Model IndoBERT (IndoLEM):")
print("="*75)
indobert_name = "indolem/indobert-base-uncased"
tokenizer_indobert = AutoTokenizer.from_pretrained(indobert_name)
model_indobert = AutoModel.from_pretrained(indobert_name, output_hidden_states=True).to(device)
model_indobert.eval() # Set ke mode evaluasi agar setiap dijalankan tidak terjadi perubahan pada bobot model
print("Info: Model IndoBERT berhasil dimuat✅")
print("="*75)

Melakukan Loading Model IndoBERT (IndoLEM):


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23379.36it/s]
BertModel LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Info: Model IndoBERT berhasil dimuat✅


Note: Model yang Diload Menggunakan Model Evaluasi Dikarnakan Kita Hanya Ingin Mengekstrak Representasinya Sehingga Komponen Dropout-nya Wajib Mati Agar Representasi yang Keluar dari Tiap Lapisan Tidak Akan Berubah-ubah Setiap Kali Dijalankan

### 2. Load Model XLM-R

In [ ]:
print("Melakukan Loading Model XLM-R:")
print("="*75)
xlmr_name = "xlm-roberta-base"
tokenizer_xlmr = AutoTokenizer.from_pretrained(xlmr_name)
model_xlmr = AutoModel.from_pretrained(xlmr_name, output_hidden_states=True).to(device)
model_xlmr.eval() # Set ke mode evaluasi agar setiap dijalankan tidak terjadi perubahan pada bobot model
print("Info: Model XLM-R berhasil dimuat✅")
print("="*75)

Melakukan Loading Model XLM-R:


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3105.87it/s]
XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Info: Model XLM-R berhasil dimuat✅


Note: Model yang Diload Menggunakan Model Evaluasi Dikarnakan Kita Hanya Ingin Mengekstrak Representasinya Sehingga Komponen Dropout-nya Wajib Mati Agar Representasi yang Keluar dari Tiap Lapisan Tidak Akan Berubah-ubah Setiap Kali Dijalankan